# 1.2 Typed State & Reducers

**Learning:** [1.2 Typed State](../../../Learning/LangGraph/01_beginner/1.2_typed_state.md)

**Goal:** Understand default state merging vs. reducers, use `add_messages` for chat history, and design state schemas that do not lose data.

**Prerequisite:** Complete `1.1_first_stategraph.ipynb`

## Setup

In [ ]:
import operator
import sys
from pathlib import Path
from typing import Annotated, TypedDict

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

## 1. The overwrite problem (no reducer)

Two nodes each return a `logs` list. Without a reducer, the **second node replaces the first**.

In [ ]:
class BadState(TypedDict):
    logs: list  # no reducer → last write wins


def node_a(state: BadState) -> BadState:
    return {"logs": ["step_a"]}


def node_b(state: BadState) -> BadState:
    return {"logs": ["step_b"]}


bad_graph = StateGraph(BadState)
bad_graph.add_node("a", node_a)
bad_graph.add_node("b", node_b)
bad_graph.add_edge(START, "a")
bad_graph.add_edge("a", "b")
bad_graph.add_edge("b", END)

bad_app = bad_graph.compile()
bad_result = bad_app.invoke({"logs": []})
print("Final logs (overwrite):", bad_result["logs"])  # only ['step_b'] — step_a lost!

## 2. Fix with a custom reducer (append)

`Annotated[list, reducer_fn]` tells LangGraph **how to merge** new values.

In [ ]:
def append_list(existing: list, new: list) -> list:
    # Reducer: called as reducer_fn(current_value, node_return_value)
    return existing + new


class GoodState(TypedDict):
    logs: Annotated[list, append_list]


def node_a_good(state: GoodState) -> GoodState:
    return {"logs": ["step_a"]}


def node_b_good(state: GoodState) -> GoodState:
    return {"logs": ["step_b"]}


good_graph = StateGraph(GoodState)
good_graph.add_node("a", node_a_good)
good_graph.add_node("b", node_b_good)
good_graph.add_edge(START, "a")
good_graph.add_edge("a", "b")
good_graph.add_edge("b", END)

good_app = good_graph.compile()
good_result = good_app.invoke({"logs": []})
print("Final logs (reducer):", good_result["logs"])  # ['step_a', 'step_b']

## 3. Reducer for counters — `operator.add`

For numeric fields, use `Annotated[int, operator.add]` to accumulate across nodes.

In [ ]:
class CounterState(TypedDict):
    total: Annotated[int, operator.add]  # each node adds to running total


def add_five(state: CounterState) -> CounterState:
    return {"total": 5}


def add_three(state: CounterState) -> CounterState:
    return {"total": 3}


counter_graph = StateGraph(CounterState)
counter_graph.add_node("five", add_five)
counter_graph.add_node("three", add_three)
counter_graph.add_edge(START, "five")
counter_graph.add_edge("five", "three")
counter_graph.add_edge("three", END)

counter_app = counter_graph.compile()
counter_app.invoke({"total": 0})  # 0 + 5 + 3 = 8

## 4. Chat state with `add_messages` (roadmap example)

Flow: `START → user → assistant → END`

Each node appends one message; history is preserved.

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]  # built-in reducer for LangChain messages
    user_name: str  # plain field — replaced on update, not appended


def user_node(state: ChatState) -> ChatState:
    # Partial update: only messages field uses reducer; user_name unchanged
    return {"messages": [HumanMessage(content=f"Hi, I'm {state['user_name']}")]}


def assistant_node(state: ChatState) -> ChatState:
    last = state["messages"][-1].content
    return {"messages": [AIMessage(content=f"Hello {state['user_name']}! You said: {last}"]}


chat_graph = StateGraph(ChatState)
chat_graph.add_node("user", user_node)
chat_graph.add_node("assistant", assistant_node)
chat_graph.add_edge(START, "user")
chat_graph.add_edge("user", "assistant")
chat_graph.add_edge("assistant", END)

chat_app = chat_graph.compile()
chat_result = chat_app.invoke({"messages": [], "user_name": "LangGraph"})

for i, msg in enumerate(chat_result["messages"]):
    print(f"{i + 1}. [{msg.type}] {msg.content}")

## 5. Partial updates — mixed fields on one state

One node updates `status` (replace), another appends to `messages` (reducer). Unmentioned fields are preserved.

In [ ]:
class MixedState(TypedDict):
    messages: Annotated[list, add_messages]
    status: str  # no reducer → replace


def set_status(state: MixedState) -> MixedState:
    return {"status": "processing"}  # messages untouched


def add_note(state: MixedState) -> MixedState:
    return {"messages": [HumanMessage(content="Working on it...")]}  # status untouched


def finish(state: MixedState) -> MixedState:
    return {
        "status": "done",
        "messages": [AIMessage(content="All finished!")],
    }


mixed_graph = StateGraph(MixedState)
mixed_graph.add_node("set_status", set_status)
mixed_graph.add_node("add_note", add_note)
mixed_graph.add_node("finish", finish)
mixed_graph.add_edge(START, "set_status")
mixed_graph.add_edge("set_status", "add_note")
mixed_graph.add_edge("add_note", "finish")
mixed_graph.add_edge("finish", END)

mixed_app = mixed_graph.compile()
mixed_result = mixed_app.invoke({"messages": [], "status": "idle"})
print("Status:", mixed_result["status"])
print("Message count:", len(mixed_result["messages"]))

## Exit Criteria Checklist

- [ ] Saw list overwrite without a reducer (`step_a` lost)
- [ ] Fixed it with `Annotated[list, append_list]`
- [ ] Used `operator.add` to accumulate an integer across nodes
- [ ] Built `ChatState` with `add_messages` — both user and assistant messages kept
- [ ] Understand partial updates: nodes return only changed fields

**Next:** [1.3 Conditional Routing](../../../Learning/LangGraph/01_beginner/1.3_conditional_routing.md) → `1.3_conditional_routing.ipynb`